## Sección 5. Comprehensions

Una *comprehension* construye una lista, un `set` o un diccionario en una sola expresión, a partir de otra colección. Su forma general es:

```python
[expresion for elemento in coleccion if condicion]           # lista
{expresion for elemento in coleccion if condicion}           # set
{clave: valor for elemento in coleccion if condicion}        # diccionario
```

Es la forma habitual de escribir en Python las operaciones de **transformar** (`SELECT`) y **filtrar** (`WHERE`) una colección.

### Ejemplos resueltos

#### Ejemplo 5.1 · Limpiar y convertir valores crudos

In [0]:
crudos = ["12.5", "8", "", "N/A", "103.75", "  44 ", "-3"]

limpios = [v.strip() for v in crudos]
print(limpios)

numericos = [float(v) for v in limpios if v.replace(".", "", 1).isdigit()]
print(numericos)

['12.5', '8', '', 'N/A', '103.75', '44', '-3']
[12.5, 8.0, 103.75, 44.0]


**Paso a paso**

1. La primera comprehension solo **transforma**: aplica `strip()` a cada valor.
2. La segunda **filtra y transforma**: se queda con los textos que representan un número y los convierte a `float`.
3. `v.replace(".", "", 1).isdigit()` quita el primer punto decimal y comprueba que lo que queda son dígitos. Por eso `"-3"` se descarta: en este ejemplo, un importe negativo no es válido.
4. La segunda comprehension equivale a este bucle:

#### Ejemplo 5.2 · Clasificar con `if-else` dentro de la comprehension

In [0]:
latencias = [120, 85, 430, 95, 1500, 88]

estado = ["LENTA" if ms > 300 else "OK" for ms in latencias]
print(estado)
print(list(zip(latencias, estado)))
print("Lentas:", estado.count("LENTA"))

['OK', 'OK', 'LENTA', 'OK', 'LENTA', 'OK']
[(120, 'OK'), (85, 'OK'), (430, 'LENTA'), (95, 'OK'), (1500, 'LENTA'), (88, 'OK')]
Lentas: 2


**Paso a paso**

1. Cuando la condición sirve para **elegir el valor**, el `if-else` va **antes** del `for`.
2. Cuando la condición sirve para **filtrar elementos**, el `if` va **después** del `for` (Ejemplo 5.1).

#### Ejemplo 5.3 · Aplanar lotes con una comprehension anidada

In [0]:
lotes = [[101, 102, 103], [104], [], [105, 106]]

registros = [r for lote in lotes for r in lote]
print(registros)
print("Total registros:", len(registros))

[101, 102, 103, 104, 105, 106]
Total registros: 6


**Paso a paso**

Los `for` se leen en el mismo orden que si fueran bucles anidados: el primero es el exterior.

```python
registros = []
for lote in lotes:          # primer for: bucle exterior
    for r in lote:          # segundo for: bucle interior
        registros.append(r)
```

#### Ejemplo 5.4 · Set de dominios normalizados

In [0]:
emails = ["ana@Acme.com", "luis@globex.com ", "marta@acme.com",
          "pablo@Initech.io", "eva@globex.com"]

dominios = {e.strip().lower().split("@")[1] for e in emails}
print(sorted(dominios))
print("Empresas distintas:", len(dominios))

['acme.com', 'globex.com', 'initech.io']
Empresas distintas: 3


**Paso a paso**

1. Para cada email: se quitan los espacios, se pasa a minúsculas y se toma lo que hay después de `@`.
2. Al usar llaves `{}`, el resultado es un `set`. `Acme.com` y `acme.com` quedan unificados tras normalizarlos.

#### Ejemplo 5.5 · Comprehensions de diccionario

In [0]:
stock = {"SKU-001": 120, "SKU-002": 0, "SKU-003": 15, "SKU-004": 0, "SKU-005": 340}

agotados = {sku: uds for sku, uds in stock.items() if uds == 0}
print(agotados)

estado = {sku: ("REPONER" if uds < 20 else "OK") for sku, uds in stock.items()}
print(estado)

columnas = ["ID Pedido", "Fecha Alta", "Importe Total"]
renombrar = {c: c.lower().replace(" ", "_") for c in columnas}
print(renombrar)

{'SKU-002': 0, 'SKU-004': 0}
{'SKU-001': 'OK', 'SKU-002': 'REPONER', 'SKU-003': 'REPONER', 'SKU-004': 'REPONER', 'SKU-005': 'OK'}
{'ID Pedido': 'id_pedido', 'Fecha Alta': 'fecha_alta', 'Importe Total': 'importe_total'}


**Paso a paso**

1. `stock.items()` entrega los pares `(clave, valor)`, que se desempaquetan en `sku, uds`.
2. El primer diccionario **filtra**, el segundo **transforma los valores** y el tercero crea un mapeo para **renombrar columnas** a *snake_case*.

#### Ejemplo 5.6 · Transformaciones encadenadas al estilo de Spark

Las transformaciones básicas de Spark (`map`, `filter`, `flatMap`, `distinct`) tienen un equivalente directo en las comprehensions. Vamos a procesar un micro-lote de logs con el formato `timestamp|usuario|nivel|etiquetas`.

In [0]:
lineas = [
    "2026-10-15T08:00:01|u17|ERROR|pago,timeout",
    "2026-10-15T08:00:02|u03|INFO|login",
    "2026-10-15T08:00:04|u17|ERROR|pago,tarjeta",
    "2026-10-15T08:00:05|u21|WARN|latencia",
    "2026-10-15T08:00:07|u08|ERROR|stock",
]

# map: cada línea se convierte en una lista de campos
registros = [linea.split("|") for linea in lineas]
print("Registros:", len(registros))
print("Primero:", registros[0])

# filter: solo los errores
errores = [r for r in registros if r[2] == "ERROR"]
print("Errores:", len(errores))

# flatMap: cada error produce tantas etiquetas como tenga
etiquetas = [etiqueta for r in errores for etiqueta in r[3].split(",")]
print("Etiquetas:", etiquetas)

# distinct: usuarios afectados
usuarios = {r[1] for r in errores}
print("Usuarios con errores:", sorted(usuarios))

Registros: 5
Primero: ['2026-10-15T08:00:01', 'u17', 'ERROR', 'pago,timeout']
Errores: 3
Etiquetas: ['pago', 'timeout', 'pago', 'tarjeta', 'stock']
Usuarios con errores: ['u08', 'u17']


**Paso a paso**

| Paso | Comprehension | Equivalente en Spark |
| --- | --- | --- |
| Convertir cada elemento | `[f(x) for x in datos]` | `map` |
| Quedarse con algunos elementos | `[x for x in datos if cond]` | `filter` |
| Un elemento produce varios | `[y for x in datos for y in f(x)]` | `flatMap` |
| Eliminar repetidos | `{x for x in datos}` | `distinct` |
1. `registros` aplica `split("|")` a cada línea: de 5 textos pasamos a 5 listas de campos.
2. `errores` filtra por el campo 2 (nivel) y se queda con 3 registros.
3. `etiquetas` usa dos `for`: el primero recorre los errores y el segundo recorre las etiquetas de cada error. De 3 registros salen 5 etiquetas.
4. La comprehension de `set` devuelve los usuarios sin repetir: `u17` tuvo dos errores, pero aparece una vez.

#### Ejemplo 5.7 · Repartir claves en particiones con una comprehension de diccionario

En el particionado por hash, cada clave va a la partición `hash(clave) % num_particiones`. Una comprehension de diccionario con otra de lista dentro construye todas las particiones de una vez.

In [0]:
import zlib

num_particiones = 3
claves = ["C001", "C002", "C003", "C004", "C005", "C001", "C006", "C003", "C007", "C001"]

# Una lista de claves por partición
particiones = {
    p: [c for c in claves if zlib.crc32(c.encode()) % num_particiones == p]
    for p in range(num_particiones)
}
for p, contenido in particiones.items():
    print(f"Partición{p}:{contenido}")

tamanos = {p: len(contenido) for p, contenido in particiones.items()}
print("Tamaños:", tamanos)

media = len(claves) / num_particiones
print("Factor de skew:", round(max(tamanos.values()) / media, 2))

Partición0:['C003', 'C003', 'C007']
Partición1:['C002', 'C005', 'C006']
Partición2:['C001', 'C004', 'C001', 'C001']
Tamaños: {0: 3, 1: 3, 2: 4}
Factor de skew: 1.2


**Paso a paso**

1. La comprehension exterior crea una clave por partición (`for p in range(num_particiones)`).
2. Para cada partición, la comprehension interior recorre todas las claves y se queda con las que tienen ese resto. `zlib.crc32()` da siempre el mismo número para la misma clave, así que `C001` cae siempre en la misma partición.
3. Una comprehension se puede escribir en varias líneas para que sea más legible: Python sabe que continúa hasta que se cierran las llaves.
4. `tamanos` se construye con otra comprehension sobre `particiones.items()`, y con ella se calcula el factor de skew (máximo / media).
5. Esta versión recorre la lista de claves una vez por partición. Con millones de claves sería más eficiente un bucle que recorra la lista una sola vez; la comprehension aporta claridad, pero no siempre es la opción más rápida.

### Ejercicios propuestos

#### Ejercicio 5.1 · Proyección y filtro de registros
Con comprehensions:

1. Crea una lista de diccionarios que contenga solo `id` e `importe` de los registros con importe mayor que 100.
2. Calcula el total de esos importes.
3. Obtén el `set` de canales distintos y muéstralo ordenado.

In [0]:
registros = [
    {"id": 1, "cliente": "Acme", "importe": 250.0, "canal": "web"},
    {"id": 2, "cliente": "Globex", "importe": 80.0, "canal": "tienda"},
    {"id": 3, "cliente": "Initech", "importe": 1200.0, "canal": "web"},
    {"id": 4, "cliente": "Umbrella", "importe": 99.99, "canal": "app"},
    {"id": 5, "cliente": "Acme", "importe": 430.0, "canal": "app"},
]

filtrados = [{"id": r["id"], "importe": r["importe"]} for r in registros if r["importe"] > 100]
total = sum(r["importe"] for r in filtrados)
canales = sorted({r["canal"] for r in registros})

print(filtrados)
print("Total:", total)
print("Canales:", canales)

[{'id': 1, 'importe': 250.0}, {'id': 3, 'importe': 1200.0}, {'id': 5, 'importe': 430.0}]
Total: 1880.0
Canales: ['app', 'tienda', 'web']


#### Ejercicio 5.2 · Transponer filas a columnas
Con una comprehension anidada, construye `columnas`: una lista con una lista por cada columna. No uses `zip()`. Después suma las unidades, que son la columna 1.

In [0]:
filas = [
    [1, 10, 2.5, 0.0],
    [2, 4, 10.0, 0.1],
    [3, 7, 3.2, 0.0],
]

columnas = [[fila[c] for fila in filas] for c in range(len(filas[0]))]
unidades_totales = sum(columnas[1])

print(columnas)
print("Unidades totales:", unidades_totales)

[[1, 2, 3], [10, 4, 7], [2.5, 10.0, 3.2], [0.0, 0.1, 0.0]]
Unidades totales: 21


#### Ejercicio 5.3 · Inferir el esquema a partir de un registro de muestra
Con una comprehension de diccionario, construye `columna → nombre del tipo` (usa `type(v).__name__`). Después obtén con una comprehension de lista las columnas cuyo tipo no se puede inferir porque el valor es `None`.

In [0]:
muestra = {
    "id": 1043,
    "cliente": "Acme",
    "importe": 99.9,
    "activo": True,
    "fecha": "2026-10-01",
    "descuento": None,
}

esquema = {k: type(v).__name__ for k, v in muestra.items()}
no_inferibles = [k for k, v in muestra.items() if v is None]

print(esquema)
print("No se puede inferir:", no_inferibles)

{'id': 'int', 'cliente': 'str', 'importe': 'float', 'activo': 'bool', 'fecha': 'str', 'descuento': 'NoneType'}
No se puede inferir: ['descuento']


#### Ejercicio 5.4 · Generar las rutas de particiones
Con una sola comprehension, genera las rutas con el formato `ventas/year=AAAA/month=MM/day=DD`. El mes y el día deben tener dos dígitos: usa `{m:02d}` en un f-string.


In [0]:
anio = 2026
meses = [9, 10]
dias = [1, 2, 3]

rutas = [f"ventas/year={anio}/month={m:02d}/day={d:02d}" for m in meses for d in dias]

for ruta in rutas:
    print(ruta)
print("Particiones:", len(rutas))

ventas/year=2026/month=09/day=01
ventas/year=2026/month=09/day=02
ventas/year=2026/month=09/day=03
ventas/year=2026/month=10/day=01
ventas/year=2026/month=10/day=02
ventas/year=2026/month=10/day=03
Particiones: 6


#### Ejercicio 5.5 · Reescribir un bucle como comprehension

Este código guarda la duración de los jobs que no terminaron en error:
Reescríbelo como **una sola comprehension de diccionario**. Pista: puedes llamar varias veces a `linea.split(";")`.

In [0]:
lineas = ["job_a;120;OK", "job_b;45;ERROR", "job_c;300;OK", "job_d;12;OK"]

resultado = {
    linea.split(";")[0]: int(linea.split(";")[1])
    for linea in lineas
    if linea.split(";")[2] != "ERROR"
}

print(resultado)

{'job_a': 120, 'job_c': 300, 'job_d': 12}


#### Ejercicio 5.6 · Extraer las columnas de partición de las rutas

En un data lake con particionado estilo Hive, la ruta de cada fichero contiene los valores de las columnas de partición con el formato `columna=valor`.
Resuelve cada paso con comprehensions:

1. Construye `metadatos`: una lista con un diccionario por ruta que contenga solo los segmentos `columna=valor`. Muestra el primero y el último.
2. Obtén las particiones distintas como tuplas `(mes, día)`, ordenadas.
3. Cuenta cuántos ficheros pertenecen al mes `"10"`. Pista: `zip(rutas, metadatos)` recorre las dos listas a la vez.
4. Construye un diccionario `"mes-día" → número de ficheros`.

In [0]:
rutas = [
    "ventas/year=2026/month=09/day=30/part-0000.parquet",
    "ventas/year=2026/month=10/day=01/part-0000.parquet",
    "ventas/year=2026/month=10/day=01/part-0001.parquet",
    "ventas/year=2026/month=10/day=02/part-0000.parquet",
    "ventas/year=2026/month=10/day=02/part-0001.parquet",
    "ventas/year=2026/month=10/day=02/part-0002.parquet",
]

# 1. Metadatos como diccionario por ruta
metadatos = [
    {seg.split("=")[0]: seg.split("=")[1] for seg in ruta.split("/") if "=" in seg}
    for ruta in rutas
]
print(metadatos[0])
print(metadatos[-1])

# 2. Particiones distintas ordenadas
particiones_distintas = sorted({(m["month"], m["day"]) for m in metadatos})
print("Particiones distintas:", particiones_distintas)

# 3. Ficheros del mes 10
ficheros_octubre = sum(1 for _, m in zip(rutas, metadatos) if m["month"] == "10")
print(f"Ficheros de octubre: {ficheros_octubre} de {len(rutas)}")

# 4. Ficheros por partición
ficheros_por_particion = {
    f"{mes}-{dia}": sum(1 for m in metadatos if m["month"] == mes and m["day"] == dia)
    for mes, dia in particiones_distintas
}
print("Ficheros por partición:", ficheros_por_particion)

{'year': '2026', 'month': '09', 'day': '30'}
{'year': '2026', 'month': '10', 'day': '02'}
Particiones distintas: [('09', '30'), ('10', '01'), ('10', '02')]
Ficheros de octubre: 5 de 6
Ficheros por partición: {'09-30': 1, '10-01': 2, '10-02': 3}


#### Ejercicio 5.7 · *Explode* de pedidos con líneas anidadas

La función `explode()` de Spark convierte un registro con un array en varias filas, una por cada elemento del array. Vamos a simularla.
1. Con una comprehension de dos `for`, genera una fila `(id_pedido, sku, uds)` por cada línea de pedido y muéstralas.
2. Comprueba qué pedidos desaparecen en el *explode* porque no tienen líneas.
3. Con una comprehension de `set`, obtén los SKU distintos. Después, con una comprehension de diccionario, calcula las unidades totales de cada SKU.

In [0]:
pedidos = [
    {"id": "P1", "lineas": [{"sku": "A1", "uds": 2}, {"sku": "B7", "uds": 1}]},
    {"id": "P2", "lineas": [{"sku": "A1", "uds": 5}]},
    {"id": "P3", "lineas": []},
    {"id": "P4", "lineas": [{"sku": "C3", "uds": 4}, {"sku": "B7", "uds": 3}, {"sku": "A1", "uds": 1}]},
]

# 1. Explode aplanando a tuplas (id_pedido, sku, uds)
filas_explode = [
    (p["id"], linea["sku"], linea["uds"])
    for p in pedidos
    for linea in p["lineas"]
]
for fila in filas_explode:
    print(fila)
print("Filas tras el explode:", len(filas_explode))

# 2. Pedidos sin líneas
sin_lineas = [p["id"] for p in pedidos if not p["lineas"]]
print("Pedidos sin líneas:", sin_lineas)

# 3. SKU únicos y suma de unidades
skus = {f[1] for f in filas_explode}
unidades_por_sku = {sku: sum(f[2] for f in filas_explode if f[1] == sku) for sku in skus}
print("Unidades por SKU:", unidades_por_sku)

('P1', 'A1', 2)
('P1', 'B7', 1)
('P2', 'A1', 5)
('P4', 'C3', 4)
('P4', 'B7', 3)
('P4', 'A1', 1)
Filas tras el explode: 6
Pedidos sin líneas: ['P3']
Unidades por SKU: {'C3': 4, 'A1': 8, 'B7': 4}
